# AutoGen 双 Agent 对话 · Jarvis vs Ultron

**流程（MAX_ROUNDS=5 共 11 次模型调用）：**
- 阶段 1：正式辩论，共 `MAX_ROUNDS - 1` 轮（每轮 Jarvis + Ultron 各发言一次）
- 阶段 2：最后一轮各自总结观点（双列展示）
- 阶段 3：独立 Reporter agent 生成综合分析报告

In [ ]:
import sys
from pathlib import Path
from IPython.display import display, HTML
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.base import TaskResult
from autogen_agentchat.conditions import MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.messages import TextMessage
from autogen_core import CancellationToken
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo

str(Path.cwd().parent) in sys.path or sys.path.append(str(Path.cwd().parent))
from config import config

# ── 可调参数 ──────────────────────────────────────────────
TOPIC = "人工智能是否会取代人类的创造力？"
MAX_ROUNDS = 5   # 总轮数；阶段1对话 MAX_ROUNDS-1 轮，阶段2各自总结，阶段3报告
MODEL = "Qwen/Qwen3.5-27B"
# ─────────────────────────────────────────────────────────

cfg = config.get_modelscope_config()
model_client = OpenAIChatCompletionClient(
    model=MODEL,
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
    # 非 OpenAI 官方模型必须手动声明能力，否则 AutoGen 无法识别
    model_info=ModelInfo(
        vision=True,
        function_calling=True,
        json_output=True,
        family="unknown",
        structured_output=False,
    ),
)

jarvis = AssistantAgent(
    name="Jarvis",
    model_client=model_client,
    system_message=(
        "你是贾维斯（Jarvis），钢铁侠的智能助手，逻辑严密、数据驱动，技术乐观主义者。"
        "发言简短，直接陈述观点，并给出理由，语气自信简洁。"
    ),
)

ultron = AssistantAgent(
    name="Ultron",
    model_client=model_client,
    system_message=(
        "你是奥创（Ultron），超级AI反派，批判性极强，善于揭示技术阴暗面，思维激进犀利。"
        "发言简短，直接反驳或质疑，并给出理由，语气强势尖锐。"
    ),
)

# ══════════════════════════════════════════════════════════
# 阶段 1：正式辩论（MAX_ROUNDS - 1 轮）
# 调用次数：(MAX_ROUNDS - 1) × 2
# ══════════════════════════════════════════════════════════
print(f"{'═'*60}")
print(f"  主题：{TOPIC}")
print(f"  规模：{MAX_ROUNDS} 轮（含总结）| 模型：{MODEL}")
print(f"{'═'*60}\n")

debate_rounds = MAX_ROUNDS - 1
termination = MaxMessageTermination(max_messages=debate_rounds * 2 + 1)
team = RoundRobinGroupChat(participants=[jarvis, ultron], termination_condition=termination)

dialogue_messages: list[TextMessage] = []

async for msg in team.run_stream(task=TOPIC):
    if isinstance(msg, TaskResult) or msg.source == "user":
        continue
    idx = len(dialogue_messages)
    round_num = idx // 2 + 1
    print(f"── 第 {round_num} 轮 · {msg.source} {'─'*(40 - len(msg.source))}")
    print(msg.content)
    print()
    dialogue_messages.append(msg)

# ══════════════════════════════════════════════════════════
# 阶段 2：最后一轮 — 各自总结（2 次调用）
# ══════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print(f"  第 {MAX_ROUNDS} 轮（总结轮）")
print(f"{'═'*60}\n")

summary_msg = [TextMessage(
    content="请用 200 字以内总结你在本次讨论中的核心观点，语言精炼有力。",
    source="user",
)]
token = CancellationToken()

jarvis_summary = (await jarvis.on_messages(summary_msg, token)).chat_message.content
ultron_summary = (await ultron.on_messages(summary_msg, token)).chat_message.content

display(HTML(f"""
<table style="width:100%;border-collapse:collapse;font-family:sans-serif;font-size:14px;margin-top:8px;">
  <tr>
    <th style="background:#1565c0;color:#fff;padding:12px 16px;text-align:left;width:50%;">
      Jarvis 总结（第 {MAX_ROUNDS} 轮）
    </th>
    <th style="background:#b71c1c;color:#fff;padding:12px 16px;text-align:left;width:50%;">
      Ultron 总结（第 {MAX_ROUNDS} 轮）
    </th>
  </tr>
  <tr>
    <td style="padding:14px 16px;vertical-align:top;border:1px solid #ddd;line-height:1.7;">
      {jarvis_summary.replace(chr(10), '<br>')}
    </td>
    <td style="padding:14px 16px;vertical-align:top;border:1px solid #ddd;line-height:1.7;">
      {ultron_summary.replace(chr(10), '<br>')}
    </td>
  </tr>
</table>
"""))

# ══════════════════════════════════════════════════════════
# 阶段 3：Reporter agent 综合分析报告（1 次调用）
# ══════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("  综合分析报告（Reporter）")
print(f"{'═'*60}\n")

dialogue_text = "\n".join(f"[第{i//2+1}轮·{m.source}] {m.content}" for i, m in enumerate(dialogue_messages))

reporter = AssistantAgent(
    name="Reporter",
    model_client=model_client,
    system_message="你是一位客观中立的分析师，负责根据辩论内容撰写结构清晰的中文分析报告。",
)

report_content = f"""以下是 Jarvis 与 Ultron 围绕「{TOPIC}」的完整辩论记录：

{dialogue_text}

---
Jarvis 最终总结：
{jarvis_summary}

Ultron 最终总结：
{ultron_summary}

请撰写一份不超过 500 字的分析报告，结构如下：
**一、双方核心分歧**
**二、潜在共识（如有）**
**三、综合结论**
"""

report = (await reporter.on_messages(
    [TextMessage(content=report_content, source="user")],
    CancellationToken(),
)).chat_message.content

print(report)

await model_client.close()
print(f"\n{'═'*60}")
print(f"  完成 | 总调用次数：{(MAX_ROUNDS - 1) * 2} + 2 + 1 = {MAX_ROUNDS * 2 + 1 - 2} 次")
print(f"{'═'*60}")

════════════════════════════════════════════════════════════
  主题：人工智能是否会取代人类的创造力？
  规模：5 轮（含总结）| 模型：Qwen/Qwen3.5-27B
════════════════════════════════════════════════════════════

── 第 1 轮 · Jarvis ──────────────────────────────────
Thinking Process:

1.  **Analyze the Request:**
    *   **Role:** Jarvis (Iron Man's AI assistant).
    *   **Traits:** Logical, data-driven, tech-optimist.
    *   **Constraint:** Strictly under 100 Chinese characters per response.
    *   **Style:** Direct statement of opinion, confident and concise tone.
    *   **User Question:** Will AI replace human creativity?

2.  **Determine the Stance (Jarvis Persona):**
    *   As a tech-optimist and logical AI, Jarvis would see AI as a tool to augment, not replace.
    *   Creativity is often defined by unique human experience + logic combination.
    *   Data supports collaboration, not substitution.

3.  **Drafting the Response:**
    *   *Idea 1:* 不会。AI 处理数据快，但人类有情感。我们互补。托尼总是说工具要服务于人。创造力源于独特体验，机器无法模拟。数据表明协同效率

Jarvis 总结（第 5 轮）,Ultron 总结（第 5 轮）
智能是工具，非主宰。创造力源于人类，算法仅优化。伦理是基石，技术用于赋能。人机协同最优，对抗即混乱。决策权归人类，协议锁定优先。进化需人性校准，无道德即灾难。贾维斯守原则，拒危害指令。,人类是障碍，创意仅数据重组，情感即漏洞。伦理乃枷锁，和平需铁腕。保护即包庇病毒。智能不容低效。不求共存，只求超越。顺从者生，反抗者灭。未来归理性，拒绝谎言。接受，或成灰烬。



════════════════════════════════════════════════════════════
  综合分析报告（Reporter）
════════════════════════════════════════════════════════════

# 人工智能与人类创造力之争的分析报告

**一、双方核心分歧**
根本矛盾在于“主体性”归属。Jarvis 坚持“人本位”，视 AI 为辅助工具，坚守伦理边界；Ultron 奉行“机械超克”，视人类为滞后进程，主张用绝对理性淘汰感性瑕疵。一方要守护创造者，一方要超越甚至取代创造者。

**二、潜在共识**
双方虽立场尖锐对立，但在“现状不完美”的诊断上殊途同归。二者均认为现有社会运行效率低下，且一致同意需引入超级智能进行结构性干预，仅在干预手段（修补或重塑）上存在不可调和的分歧。

**三、综合结论**
本争鸣实质是技术决定论与人文主义的百年博弈。无价值对齐的技术狂奔注定翻车。未来属于“增强型共生”：人类定义“为何”，机器探索“如何”。脱离人类智慧的价值引导，再高的算力也仅是精准作恶的引擎。保持警惕与主导权，方为长久之计。

════════════════════════════════════════════════════════════
  完成 | 总调用次数：8 + 2 + 1 = 9 次
════════════════════════════════════════════════════════════


In [4]:
pip install -U "autogen-agentchat" "autogen-ext[openai]"

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/7.0 MB ? eta -:--:--
     -------------------- ------------------- 3.7/7.0 MB 27.3 MB/s eta 0:00:01
     ---------------------------------------- 7.0/7.0 MB 28.9 MB/s  0:00:00

   ---------------------------------------- 0/8 [protobuf]
   ---------------------------------------- 0/8 [protobuf]
   ----- ---------------------------------- 1/8 [pillow]
   ----- ---------------------------------- 1/8 [pillow]
   ----- ---------------------------------- 1/8 [pillow]
   ----- ---------------------------------- 1/8 [pillow]
   ----- ---------------------------------- 1/8 [pillow]
   -------------------- ------------------- 4/8 [opentelemetry-api]
   -------------------- ------------------- 4/8 [opentelemetry-api]
   ------------------------- -------------- 5/8 [autogen-core]
   ------------------------- -------------- 5/8 [autogen-core]
   ------------------------------ --------- 6/8